In [2]:
import numpy as np

In [3]:
def _build(signal, target_weight, target_shares, current_shares, price, net_liq,
           commission_bps, slippage_bps, min_fee, min_trade_pct, min_trade_dollars, reason=None):
    delta_shares = target_shares - current_shares
    trade_dollars = abs(delta_shares) * price

    # Umbral: no operar si el ajuste es pequeño
    skipped = False
    if delta_shares != 0:
        too_small_pct = (trade_dollars / net_liq) < min_trade_pct
        too_small_usd = trade_dollars < min_trade_dollars
        if too_small_pct or too_small_usd:
            target_shares, delta_shares, trade_dollars = current_shares, 0, 0.0
            skipped = True

    # Costes: comisión (con mínimo por orden) + slippage
    if trade_dollars > 0:
        commission = max(trade_dollars * commission_bps / 1e4, min_fee)
        slippage = trade_dollars * slippage_bps / 1e4
    else:
        commission = slippage = 0.0

    action = "BUY" if delta_shares > 0 else "SELL" if delta_shares < 0 else "HOLD"

    return {
        "momentum_signal": signal,
        "target_weight": target_weight,
        "target_dollars": target_shares * price,
        "target_shares": target_shares,
        "current_shares": current_shares,
        "delta_shares": delta_shares,
        "trade_dollars": trade_dollars,
        "est_commission": commission,
        "est_slippage": slippage,
        "est_total_cost": commission + slippage,
        "skipped_by_threshold": skipped,
        "action": action,
        "reason": reason,
    }


def momentum(prices, lookback, scale, allocation_pct, net_liq, current_price, current_shares,
             commission_bps=1.0,        # comisión en puntos básicos sobre el nominal
             slippage_bps=2.0,          # deslizamiento estimado en pb
             min_fee=1.0,               # comisión mínima por orden ($)
             min_trade_pct=0.005,       # umbral: 0.5% del net_liq
             min_trade_dollars=0.0):    # umbral absoluto opcional ($)

    cost_args = (commission_bps, slippage_bps, min_fee, min_trade_pct, min_trade_dollars)

    prices = np.asarray(prices, dtype=float)
    clean_prices = prices[np.isfinite(prices)]

    if net_liq <= 0 or current_price <= 0:
        return _build(0.0, 0.0, current_shares, current_shares, max(current_price, 1e-9),
                      max(net_liq, 1e-9), *cost_args, reason="net_liq o precio no válido")

    if len(clean_prices) <= lookback:
        return _build(0.0, 0.0, 0, current_shares, current_price, net_liq,
                      *cost_args, reason="historial insuficiente")

    past_price = clean_prices[-lookback - 1]
    last_price = clean_prices[-1]

    if past_price <= 0 or last_price <= 0:
        return _build(0.0, 0.0, 0, current_shares, current_price, net_liq,
                      *cost_args, reason="precio no válido")

    r = (last_price - past_price) / past_price
    signal = float(np.tanh(r * scale))
    target_weight = float(np.clip(signal * allocation_pct / 100, -1, 1))
    target_shares = int(round(target_weight * net_liq / current_price))

    return _build(signal, target_weight, target_shares, current_shares, current_price, net_liq, *cost_args)

In [5]:
TRADING_DAYS = 252


# ---------- Datos sintéticos ----------
def gen_regime_prices(n_days, rng, p0=100.0, vol=0.15, drift_ann=0.25, mean_regime_days=120):
    """GBM con regímenes de deriva alternante (alcista/bajista)."""
    dt = 1 / TRADING_DAYS
    rets = np.empty(n_days)
    i, sign = 0, rng.choice([-1, 1])
    while i < n_days:
        length = max(20, int(rng.exponential(mean_regime_days)))
        j = min(n_days, i + length)
        mu = sign * drift_ann
        rets[i:j] = (mu - 0.5 * vol**2) * dt + vol * np.sqrt(dt) * rng.standard_normal(j - i)
        sign *= -1
        i = j
    return p0 * np.exp(np.concatenate([[0.0], np.cumsum(rets)]))


def gen_random_walk(n_days, rng, p0=100.0, vol=0.20, drift_ann=0.05):
    """GBM puro: sin tendencias explotables."""
    dt = 1 / TRADING_DAYS
    rets = (drift_ann - 0.5 * vol**2) * dt + vol * np.sqrt(dt) * rng.standard_normal(n_days)
    return p0 * np.exp(np.concatenate([[0.0], np.cumsum(rets)]))


# ---------- Backtest ----------
def run_backtest(prices, lookback=60, scale=5.0, allocation_pct=100.0, capital=100_000.0,
                 commission_bps=1.0, slippage_bps=2.0, min_fee=1.0,
                 min_trade_pct=0.005, min_trade_dollars=0.0, rebalance_every=1):
    cash, shares = capital, 0
    equity = np.full(len(prices), np.nan)
    total_cost, traded, n_trades = 0.0, 0.0, 0

    for t in range(len(prices) - 1):
        px = prices[t]
        net_liq = cash + shares * px
        equity[t] = net_liq

        if t >= lookback and t % rebalance_every == 0:
            res = momentum(prices[max(0, t - lookback - 1): t + 1], lookback, scale, allocation_pct,
                           net_liq, px, shares, commission_bps, slippage_bps, min_fee,
                           min_trade_pct, min_trade_dollars)
            d = res["delta_shares"]
            if d != 0:
                # ejecución al cierre del día siguiente (evita look-ahead)
                fill = prices[t + 1] * (1 + np.sign(d) * slippage_bps / 1e4)
                notional = abs(d) * fill
                fee = max(notional * commission_bps / 1e4, min_fee)
                cash -= d * fill + fee
                shares += d
                total_cost += fee + abs(d) * prices[t + 1] * slippage_bps / 1e4
                traded += notional
                n_trades += 1

    equity[-1] = cash + shares * prices[-1]
    equity = equity[~np.isnan(equity)]
    r = np.diff(equity) / equity[:-1]
    ann_ret = (equity[-1] / equity[0]) ** (TRADING_DAYS / len(r)) - 1
    ann_vol = r.std() * np.sqrt(TRADING_DAYS)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0.0
    dd = (equity / np.maximum.accumulate(equity) - 1).min()
    years = len(r) / TRADING_DAYS
    return {
        "final": equity[-1], "ann_ret": ann_ret, "ann_vol": ann_vol, "sharpe": sharpe,
        "max_dd": dd, "n_trades": n_trades, "turnover_x": traded / capital / years,
        "cost_pct": total_cost / capital, "equity": equity,
    }


def buy_and_hold(prices, capital=100_000.0):
    eq = capital * prices / prices[0]
    r = np.diff(eq) / eq[:-1]
    ann_ret = (eq[-1] / eq[0]) ** (TRADING_DAYS / len(r)) - 1
    ann_vol = r.std() * np.sqrt(TRADING_DAYS)
    return {"ann_ret": ann_ret, "sharpe": ann_ret / ann_vol, "max_dd": (eq / np.maximum.accumulate(eq) - 1).min()}


def summarize(label, gen, thresholds, n_paths=100, n_days=5 * TRADING_DAYS, seed=42):
    print(f"\n=== {label} ({n_paths} trayectorias, {n_days // TRADING_DAYS} años) ===")
    rng = np.random.default_rng(seed)
    paths = [gen(n_days, rng) for _ in range(n_paths)]

    bh = [buy_and_hold(p) for p in paths]
    print(f"Buy&Hold        ret {np.mean([b['ann_ret'] for b in bh]):7.1%}  "
          f"sharpe {np.mean([b['sharpe'] for b in bh]):5.2f}  maxDD {np.mean([b['max_dd'] for b in bh]):7.1%}")

    print(f"{'umbral':>8} {'ret/año':>8} {'sharpe':>7} {'maxDD':>7} {'trades':>7} {'turnover':>9} {'costes%':>8}")
    for th in thresholds:
        res = [run_backtest(p, min_trade_pct=th) for p in paths]
        m = lambda k: np.mean([x[k] for x in res])
        print(f"{th:>8.1%} {m('ann_ret'):>8.1%} {m('sharpe'):>7.2f} {m('max_dd'):>7.1%} "
              f"{m('n_trades'):>7.0f} {m('turnover_x'):>8.1f}x {m('cost_pct'):>8.2%}")


if __name__ == "__main__":
    thresholds = [0.0, 0.005, 0.02, 0.05, 0.10]
    summarize("Regímenes de tendencia", gen_regime_prices, thresholds)
    summarize("Paseo aleatorio (sin tendencia)", gen_random_walk, thresholds)

    print("\n=== Regímenes con costes altos (10 pb comisión + 10 pb slippage) ===")
    rng = np.random.default_rng(7)
    paths = [gen_regime_prices(5 * TRADING_DAYS, rng) for _ in range(100)]
    print(f"{'umbral':>8} {'ret/año':>8} {'sharpe':>7} {'trades':>7} {'costes%':>8}")
    for th in thresholds:
        res = [run_backtest(p, min_trade_pct=th, commission_bps=10, slippage_bps=10) for p in paths]
        m = lambda k: np.mean([x[k] for x in res])
        print(f"{th:>8.1%} {m('ann_ret'):>8.1%} {m('sharpe'):>7.2f} {m('n_trades'):>7.0f} {m('cost_pct'):>8.2%}")


=== Regímenes de tendencia (100 trayectorias, 5 años) ===
Buy&Hold        ret   -0.5%  sharpe -0.04  maxDD  -42.1%
  umbral  ret/año  sharpe   maxDD  trades  turnover  costes%
    0.0%     3.1%    0.51  -10.4%    1192     12.2x    2.45%
    0.5%     3.1%    0.51  -10.4%    1115     12.1x    2.37%
    2.0%     3.2%    0.52  -10.3%     869     11.5x    2.07%
    5.0%     3.3%    0.54  -10.2%     498      9.4x    1.50%
   10.0%     3.3%    0.55  -10.1%     218      6.5x    0.97%

=== Paseo aleatorio (sin tendencia) (100 trayectorias, 5 años) ===
Buy&Hold        ret    2.5%  sharpe  0.12  maxDD  -38.8%
  umbral  ret/año  sharpe   maxDD  trades  turnover  costes%
    0.0%    -0.8%   -0.13  -20.5%    1192     14.2x    2.70%
    0.5%    -0.7%   -0.13  -20.5%    1133     14.2x    2.64%
    2.0%    -0.7%   -0.12  -20.4%     939     13.8x    2.41%
    5.0%    -0.6%   -0.10  -20.1%     616     12.0x    1.91%
   10.0%    -0.4%   -0.09  -19.6%     311      9.0x    1.35%

=== Regímenes con costes a